# Exercise 3: Neural networks in PyTorch

In this exercise you’ll implement small neural-network building blocks from scratch and use them to train a simple classifier.

You’ll cover:
- **Basic layers**: Linear, Embedding, Dropout
- **Normalization**: LayerNorm and RMSNorm
- **MLPs + residual**: composing layers into deeper networks
- **Classification**: generating a learnable dataset, implementing cross-entropy from logits, and writing a minimal training loop

As before: fill in all `TODO`s without changing function names or signatures.
Use small sanity checks and compare to PyTorch reference implementations when useful.

In [18]:
from __future__ import annotations

import torch
from torch import nn

## Basic layers

In this section you’ll implement a few core layers that appear everywhere:

### `Linear`
A fully-connected layer that follows nn.Linear conventions:  
`y = x @ Wᵀ + b`

Important details:
- Parameters should be registered as `nn.Parameter`
- Store weight as (out_features, in_features) like nn.Linear.
- The forward pass should support leading batch dimensions: `x` can be shape `(..., in_features)`

### `Embedding`
An embedding table maps integer ids to vectors:
- input: token ids `idx` of shape `(...,)`
- output: vectors of shape `(..., embedding_dim)`

This is essentially a learnable lookup table.

### `Dropout`
Dropout randomly zeroes activations during training to reduce overfitting.
Implementation details:
- Only active in `model.train()` mode
- In training: drop with probability `p` and scale the kept values by `1/(1-p)` so the expected value stays the same
- In eval: return the input unchanged

## Instructions
- Do not use PyTorch reference modules for the parts you implement (e.g. don’t call nn.Linear inside your Linear).
- You may use standard tensor ops that you learned before (matmul, sum, mean, rsqrt, indexing, etc.).
- Use a parameter initialization method of your choice. We recommend something like Xavier-uniform.


In [19]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        # TODO: initialize parameters
        bound =  (6 / (in_features + out_features)) ** 0.5 # we use xavier uniform
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        
        with torch.no_grad():
            self.weight.uniform_(-bound, bound)
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
            with torch.no_grad():
                self.bias.zero_()
        else:
            self.bias = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., in_features)
        return: (..., out_features)
        """
        # TODO: implement
        out = x @ self.weight.T
        if self.bias is not None:
            out += self.bias
        return out
    
x = torch.randn(4, 5)
linear = Linear(5, 3)
out_torch = nn.functional.linear(x, linear.weight, linear.bias)
out = linear.forward(x)
print(out_torch == out)

tensor([[True, True, True],
        [True, True, True],
        [True, True, True],
        [True, True, True]])


In [20]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        # TODO: initialize
        bound = (6 / (num_embeddings + embedding_dim)) ** 0.5 # we use xavier uniform
        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim))
        with torch.no_grad():
            self.weight.uniform_(-bound, bound)

        

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        idx: (...,) int64
        return: (..., embedding_dim)
        """
        # TODO: implement (index into weight)
        return self.weight[idx]

In [21]:
class Dropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        In train mode: drop with prob p and scale by 1/(1-p).
        In eval mode: return x unchanged.
        """
        # TODO: implement without using nn.Dropout
        if self.training:
            mask = torch.rand(x.shape) > self.p
            return x * mask / (1 - self.p)
        else:
            return x
            


x = torch.randn(4, 5)
dropout = Dropout(0.5)
dropout.train()
out = dropout.forward(x)
out

tensor([[-1.3765, -0.0000, -0.2962, -0.6757,  0.0000],
        [ 1.9430,  0.0000,  0.3655,  0.0000,  0.0000],
        [-5.1225, -1.2192,  1.0079, -0.0000, -2.1045],
        [ 0.0918, -0.2068, -0.5113, -0.0000,  0.0000]])

## Normalization

Normalization layers help stabilize training by controlling activation statistics.

### LayerNorm
LayerNorm normalizes each example across its **feature dimension** (the last dimension):

- compute mean and variance over the last dimension
- normalize: `(x - mean) / sqrt(var + eps)`
- apply learnable per-feature scale and shift (`weight`, `bias`)

**In this exercise, assume `elementwise_affine=True` (always include `weight` and `bias`).**  
`weight` and `bias` each have shape `(D,)`.

LayerNorm is widely used in transformers because it does not depend on batch statistics.

### RMSNorm
RMSNorm is similar to LayerNorm but normalizes using only the root-mean-square:
- `x / sqrt(mean(x^2) + eps)` over the last dimension
- usually includes a learnable scale (`weight`)
- no mean subtraction

RMSNorm is popular in modern LLMs because it's faster.


In [22]:
class LayerNorm(nn.Module):
    def __init__(
        self, normalized_shape: int, eps: float = 1e-5, elementwise_affine: bool = True
    ):
        super().__init__()
        # TODO: implement
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        if elementwise_affine:
            self.weight = nn.Parameter(torch.ones(normalized_shape))
            self.bias = nn.Parameter(torch.zeros(normalized_shape))
        else:
            self.register_parameter('weight', None) # in case elementwise_affine is false but should not be in this exercise
            self.register_parameter('bias', None)
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Normalize over the last dimension.
        x: (..., D)
        """
        # TODO: implement
        var, mean = torch.var_mean(x, dim=-1, keepdim=True, unbiased=False) # only over last dim
        ret = (x - mean) / ((var + self.eps) ** 0.5)
        if self.elementwise_affine:
            ret = ret * self.weight + self.bias
        return ret
        

        
# x = torch.tensor([[1.0,1.0,2.0,2.0,2.0], [2.0,1.0,2.0,2.0,2.0]])
x = torch.randn(1, 1, 2, 2)
norm_layer = LayerNorm(x.shape[-1])
out = norm_layer.forward(x)
out_torch = nn.LayerNorm(x.shape[-1], eps=norm_layer.eps, elementwise_affine=norm_layer.elementwise_affine).forward(x)
print(out_torch)
print(out)

tensor([[[[-1.0000,  1.0000],
          [ 1.0000, -1.0000]]]], grad_fn=<NativeLayerNormBackward0>)
tensor([[[[-1.0000,  1.0000],
          [ 1.0000, -1.0000]]]], grad_fn=<AddBackward0>)


In [23]:
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-8):
        super().__init__()
        # TODO: implement
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RMSNorm: x / sqrt(mean(x^2) + eps) * weight
        over the last dimension.
        """
        # TODO: implement
        return x / ((torch.mean(x*x) + self.eps) ** 0.5) * self.weight
    
x = torch.tensor([[2.0,1.0,2.0,2.0,2.0], [2.0,1.0,2.0,2.0,2.0]])
norm_layer = RMSNorm(x.shape[-1])
out_torch = nn.RMSNorm(x.shape[-1], eps=norm_layer.eps).forward(x)
out = norm_layer.forward(x)
print(out_torch)
print(out)


tensor([[1.0847, 0.5423, 1.0847, 1.0847, 1.0847],
        [1.0847, 0.5423, 1.0847, 1.0847, 1.0847]], grad_fn=<MulBackward0>)
tensor([[1.0847, 0.5423, 1.0847, 1.0847, 1.0847],
        [1.0847, 0.5423, 1.0847, 1.0847, 1.0847]], grad_fn=<MulBackward0>)


## MLPs and residual networks

Now you’ll build larger networks by composing layers.

### MLP
An MLP is a stack of `depth` Linear layers with non-linear activations (use GELU) in between.
In this exercise you’ll support:
- configurable depth
- a hidden dimension
- optional LayerNorm between layers (a common stabilization trick)

A key skill is building networks using `nn.ModuleList` / `nn.Sequential` while keeping shapes consistent.

### Transformer-style FeedForward (FFN)
A transformer block contains a position-wise feedforward network:
- `D -> 4D -> D` (by default)
- activation is typically **GELU**

This is essentially an MLP applied independently at each token position.

### Residual wrapper
Residual connections are the simplest form of “skip connection”:
- output is `x + fn(x)`

They improve gradient flow and allow training deeper networks more reliably.

In [24]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int,
        use_layernorm: bool = False,
    ):
        super().__init__()
        # TODO: build modules (list of Linear + activation)
        # Optionally insert LayerNorm between layers.
        if depth < 2:
            raise ValueError("Cannot constuct proper MLP with depth < 2")
        layers_ = [Linear(in_dim, hidden_dim)] # we use our own
        if use_layernorm:
            layers_.append(LayerNorm(hidden_dim))
        for i in range(depth - 2):
            layers_.append(nn.GELU())
            layers_.append(Linear(hidden_dim, hidden_dim))
            if use_layernorm:
                layers_.append(LayerNorm(hidden_dim))
        layers_.append(nn.GELU())    
        layers_.append(Linear(hidden_dim, out_dim))
        self.layers = nn.Sequential(*layers_)

            
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        return self.layers(x)
    
mlp = MLP(10, 40, 10, 4, use_layernorm=True)
print(mlp.layers)

x = torch.randn([3,4,5,10])
out = mlp.forward(x)
print(out)
print(out.shape)

Sequential(
  (0): Linear()
  (1): LayerNorm()
  (2): GELU(approximate='none')
  (3): Linear()
  (4): LayerNorm()
  (5): GELU(approximate='none')
  (6): Linear()
  (7): LayerNorm()
  (8): GELU(approximate='none')
  (9): Linear()
)
tensor([[[[ 9.6326e-01, -1.0300e+00,  8.0484e-01,  1.0613e+00, -2.9234e-01,
           -1.2118e+00,  9.6178e-02,  3.6771e-01, -9.3520e-01,  7.2040e-01],
          [ 2.1433e+00,  2.2936e-02,  9.6827e-01,  1.9377e-01, -3.2542e-01,
           -6.3238e-01, -4.6265e-01,  3.9047e-01, -1.4201e+00, -1.5779e-01],
          [-8.8682e-02, -3.8950e-01,  1.4233e+00, -8.7161e-02, -6.9139e-01,
           -1.0660e+00, -3.9960e-01, -1.1927e-01, -8.7295e-01,  3.3323e-01],
          [ 1.2047e+00, -1.2766e+00,  1.2982e+00,  5.8865e-01,  3.5234e-01,
           -9.7487e-01,  6.7415e-02,  1.1476e+00, -2.2220e-01, -3.2468e-01],
          [ 1.0599e+00,  3.3828e-01,  1.7752e+00, -6.3450e-02, -9.0095e-01,
           -3.2564e-01, -8.5599e-01, -3.2604e-01, -4.9755e-01, -5.1874e-01]],

  

In [25]:
class FeedForward(nn.Module):
    """
    Transformer-style FFN: D -> 4D -> D (default)
    """

    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        # TODO: create two Linear layers and choose an activation (GELU)
        self.linear1 = Linear(d_model, d_ff)
        self.linear2 = Linear(d_ff, d_model)
        self.activation = nn.GELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: implement
        ret = self.linear1(x)
        ret = self.activation(ret)
        ret = self.linear2(ret)
        return ret

x = torch.randn([3,4,5,10])
feedforward = FeedForward(10)
out = feedforward.forward(x)
print(out)
print(out.shape)

tensor([[[[ 7.6003e-01, -5.8745e-01, -1.2515e+00, -9.1687e-02, -1.1446e-01,
            1.9811e-01, -6.9525e-01, -5.5386e-01, -2.1304e-01, -8.7476e-01],
          [ 8.6851e-02, -8.3394e-01, -2.9948e-01, -2.8951e-01,  7.5585e-02,
            5.0148e-01,  2.3417e-03,  1.4244e-01,  9.3736e-02, -2.5814e-01],
          [ 2.2645e-01,  1.5335e+00, -2.1660e-01, -2.9577e-01,  2.8032e-01,
            2.3151e-01, -1.0610e-01, -5.6733e-01,  8.3106e-01, -1.6226e+00],
          [ 3.0063e-01,  1.2610e-02, -1.6506e-01,  2.0921e-01, -5.5416e-01,
            8.7306e-02, -3.5917e-01, -1.0909e-01, -1.0811e-01,  2.8311e-01],
          [-1.3871e-03,  1.5640e-01, -1.0078e-01, -7.3056e-02,  5.9053e-01,
           -1.1213e-01, -2.0693e-01, -3.5724e-01,  1.0607e-01, -9.5279e-01]],

         [[-3.1868e-01, -8.9187e-01, -3.6821e-05, -1.5006e-01,  6.1807e-02,
            1.2300e-01,  3.8320e-01,  8.3843e-02, -1.0499e-01,  1.7716e-01],
          [ 3.7810e-01, -2.5478e-01, -9.6015e-02, -1.0685e-01, -1.1810e+00,
    

In [26]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module):
        super().__init__()
        # TODO: implement
        self.fn = fn

    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        # TODO: return x + fn(x, ...)
        return x + self.fn(x, *args, **kwargs)


## Classification problem

In this section you’ll put everything together in a minimal MNIST classification experiment.

You will:
1) download and load the MNIST dataset
2) implement cross-entropy from logits (stable, using log-softmax)
3) build a simple MLP-based classifier (flatten MNIST images first)
4) write a minimal training loop
5) report train loss curve and final accuracy

The goal here is not to reach state-of-the-art accuracy, but to understand the full pipeline:
data → model → logits → loss → gradients → parameter update.

### Model notes
- We want you to combine the MLP we implemented above with the classification head we define below into one model 

### MNIST notes
- MNIST images are `28×28` grayscale.
- After `ToTensor()`, each image has shape `(1, 28, 28)` and values in `[0, 1]`.
- For an MLP classifier, we flatten to a vector of length `784`.

## Deliverables
- Include a plot of your train loss curve in the video submission as well as a final accuracy. 
- **NOTE** Here we don't grade on model performance but we expect you to achieve at least 70% accuracy to confirm a correct model implementation.

In [27]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [28]:
transform = transforms.ToTensor()  # -> float32 in [0,1], shape (1, 28, 28)

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

# TODO: define the dataloaders
dataloader_train = DataLoader(train_ds, batch_size=8, shuffle=True)
dataloader_test = DataLoader(test_ds, batch_size=8, shuffle=False)


print(next(iter(dataloader_test))[0].shape)

torch.Size([8, 1, 28, 28])


In [29]:
def cross_entropy_from_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Compute mean cross-entropy loss from logits.

    logits: (B, C)
    targets: (B,) int64

    Requirements:
    - Use log-softmax for stability (do not use torch.nn.CrossEntropyLoss, we check this in the autograder).
    """
    # TODO: implement
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
    loss = torch.nn.functional.nll_loss(log_probs, targets)
    return loss

x = torch.tensor([[4.0, -10.0, 0.0]])
y = torch.tensor([0]) 
cross_entropy_from_logits(x, y)

tensor(0.0182)

In [30]:
class ClassificationHead(nn.Module):
    def __init__(self, d_in: int, num_classes: int):
        super().__init__()
        # TODO: implement
        self.layer = Linear(d_in, num_classes)
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., d_in)
        return: (..., num_classes) logits
        """
        # TODO: implement
        return self.layer(x)

x = torch.tensor([[4.0, -10.0, 0.0]])
head = ClassificationHead(3,2)
head(x)

tensor([[3.5973, 3.2188]], grad_fn=<AddBackward0>)

In [31]:
def accuracy(loader, model): # could not see how to do it without adding model into params
    # TODO: You can use this function to evaluate your model accuracy.
    model.eval() # not sure but i guess model is assumend to be global
    correct = 0
    total = 0

    with torch.no_grad():
        for data in loader:
            inputs, labels = data
            y_hat = model(inputs)
            _, predicted_class = torch.max(y_hat.data, 1)
            total += labels.size(0)
            correct += (predicted_class == labels).sum().item()
    
    return correct / total


In [ ]:
%run ex2.ipynb # to use own optimizer

def train_classifier(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int = 0,
) -> list[float]:
    """
    Minimal training loop for MNIST classification.

    Steps:
    - define optimizer
    - for each epoch:
        - sample minibatches
        - forward -> cross-entropy -> backward -> optimizer step
      - compute test accuracy at the end of each epoch
    - return list of training losses (one per update step)

    Requirements:
    - call model.train() during training and model.eval() during evaluation
    - do not use torch.nn.CrossEntropyLoss (use your cross_entropy_from_logits)
    """
    # TODO: implement

    torch.manual_seed(seed)

    # wanna try own optimizer
    
    params = [param for param in model.parameters() if param.requires_grad]
    state = [init_adamw_state(p) for p in params]

    # optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    

    for epoch in range(epochs):
        model.train()
        for i, batch in enumerate(train_data_loader):
            x, y = batch
            y_hat = model(x)
            loss = cross_entropy_from_logits(y_hat, y)
            
            loss.backward()
            # if i % 1000 == 0:
                # print(f"loss:  {loss.item()}")
            grads = [p.grad for p in params]
            if any(g is None for g in grads):
                print("None grads detected")
            adamw_step_many_(params, grads, state, lr=lr)

            for p in params:
                p.grad = None

            # optimizer.step()
            # optimizer.zero_grad()

        acc = accuracy(test_data_loader, model)
        print(f"Epoch {epoch}, Accuracy: {acc}")


tensor([2, 3, 4])
tensor([3, 4, 5])
tensor([1, 2, 3])
tensor([3, 4, 5])
tensor([2, 3, 4])


In [33]:

class MNIST_classifier(nn.Module):
    def __init__(self, hidden_layers, depth, use_layernorm=True):
        super().__init__()
        self.layer = nn.Sequential(
            MLP(784, hidden_layers, 64, depth, use_layernorm=use_layernorm),
            ClassificationHead(64, 10)
        )

    def forward(self, x):
        x = x.flatten(start_dim=1)
        return self.layer(x)


In [34]:
train_classifier(
    model=MNIST_classifier(96, 3),
    train_data_loader=dataloader_train,
    test_data_loader=dataloader_test,
    lr=1e-3,
    epochs=5,
)

Epoch 0, Accuracy: 0.9649
Epoch 1, Accuracy: 0.9661
Epoch 2, Accuracy: 0.9738
Epoch 3, Accuracy: 0.976
Epoch 4, Accuracy: 0.9785


In [35]:
train_classifier(
    model=MNIST_classifier(96, 3, use_layernorm=False),
    train_data_loader=dataloader_train,
    test_data_loader=dataloader_test,
    lr=1e-4,
    epochs=5,
)

Epoch 0, Accuracy: 0.9393
Epoch 1, Accuracy: 0.9567
Epoch 2, Accuracy: 0.9643
Epoch 3, Accuracy: 0.9664
Epoch 4, Accuracy: 0.9711


In [36]:
train_classifier(
    model=MNIST_classifier(64, 2, use_layernorm=False),
    train_data_loader=dataloader_train,
    test_data_loader=dataloader_test,
    lr=1e-4,
    epochs=5,
)

Epoch 0, Accuracy: 0.9186
Epoch 1, Accuracy: 0.9409
Epoch 2, Accuracy: 0.9539
Epoch 3, Accuracy: 0.9587
Epoch 4, Accuracy: 0.9633
